# P9 — Exécution et preuves sur EMR

À ouvrir dans un kernel Python sur le primaire EMR configuré suivant `docs/aws_plan.md`. Le cluster, les données S3 et le bootstrap doivent être prêts. Ce notebook ne crée pas le cluster. Il lance un calcul sur le cluster existant, donc à exécuter uniquement pendant la session AWS prévue. Sauvegarder ensuite le notebook exécuté sur S3.

In [ ]:
# Renseigne uniquement les identifiants non secrets de l'environnement européen déjà créé.
import os
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
P9_BUCKET = ''
P9_REGION = 'eu-west-3'
P9_CLUSTER_ID = ''
P9_CODE_PREFIX = 'code/COMMIT-VERIFIE'
if not P9_BUCKET or not P9_CLUSTER_ID or 'COMMIT-VERIFIE' in P9_CODE_PREFIX:
    raise ValueError('Renseigner bucket, cluster et préfixe exact du code.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOCAL = Path.home() / ('p9-run-' + RUN_ID)
LOCAL.mkdir()
OUTPUT_URI = f's3://{P9_BUCKET}/runs/{RUN_ID}'


In [ ]:
# Vérifie la région de stockage et récupère la configuration du cluster comme preuve.
if not P9_REGION.startswith('eu-'):
    raise ValueError('Le scénario exige des serveurs européens.')
def aws_json(arguments):
    return json.loads(subprocess.check_output(['aws', '--region', P9_REGION, *arguments, '--output', 'json'], text=True))
location = aws_json(['s3api', 'get-bucket-location', '--bucket', P9_BUCKET])
if location['LocationConstraint'] != P9_REGION:
    raise ValueError('Le bucket doit être dans la région européenne retenue pour le cluster.')
cluster = aws_json(['emr', 'describe-cluster', '--cluster-id', P9_CLUSTER_ID])
(LOCAL / 'cluster.json').write_text(json.dumps(cluster, indent=2))
print('Cluster :', cluster['Cluster']['Name'], cluster['Cluster']['Status']['State'])
print('Version :', cluster['Cluster']['ReleaseLabel'], 'Région :', P9_REGION)


In [ ]:
# Récupère le paquet publié et vérifie ses empreintes avant le calcul.
import hashlib
CODE_DIR = LOCAL / 'code'
subprocess.run(['aws', 's3', 'cp', f's3://{P9_BUCKET}/{P9_CODE_PREFIX}/', str(CODE_DIR), '--recursive'], check=True)
manifest = json.loads((CODE_DIR / 'manifest.json').read_text())
assert not manifest['working_tree_dirty'], 'Reconstruire le paquet depuis un commit propre.'
for filename, expected in manifest['sha256'].items():
    assert hashlib.sha256((CODE_DIR / filename).read_bytes()).hexdigest() == expected, filename
subprocess.run(['/opt/p9-venv/bin/python', '-m', 'pip', 'check'], check=True)
(LOCAL / 'environment.txt').write_text(subprocess.check_output(['/opt/p9-venv/bin/python', '-m', 'pip', 'freeze'], text=True))
print('Commit du paquet :', manifest['git_commit'])


In [ ]:
# Exécute la chaîne complète sur YARN : lecture S3, CNN, PCA et écriture directe S3.
ENV = os.environ.copy()
ENV.update({'PYSPARK_PYTHON':'/opt/p9-venv/bin/python', 'PYSPARK_DRIVER_PYTHON':'/opt/p9-venv/bin/python',
            'OMP_NUM_THREADS':'1', 'OPENBLAS_NUM_THREADS':'1', 'TF_NUM_INTRAOP_THREADS':'1', 'TF_NUM_INTEROP_THREADS':'1'})
command = ['spark-submit', '--master', 'yarn', '--deploy-mode', 'client',
           '--conf', 'spark.dynamicAllocation.enabled=false', '--num-executors', '2',
           '--executor-cores', '1', '--executor-memory', '3g', '--driver-memory', '4g',
           '--conf', 'spark.executor.memoryOverhead=3g',
           '--conf', 'spark.pyspark.python=/opt/p9-venv/bin/python']
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'TF_NUM_INTRAOP_THREADS', 'TF_NUM_INTEROP_THREADS'):
    command += ['--conf', f'spark.executorEnv.{name}=1']
command += ['--py-files', str(CODE_DIR / 'p9_src.zip'), str(CODE_DIR / 'run_pipeline.py'),
            '--input', f's3://{P9_BUCKET}/data/sample-v2/', '--output', OUTPUT_URI,
            '--max-images', '100', '--components', '20', '--batch-size', '8', '--partitions', '4']
(LOCAL / 'command.json').write_text(json.dumps(command, indent=2))
with (LOCAL / 'spark.log').open('w') as log:
    result = subprocess.run(command, env=ENV, stdout=log, stderr=subprocess.STDOUT)
print((LOCAL / 'spark.log').read_text()[-12000:])
result.check_returncode()


In [ ]:
# Lit les métriques validées depuis S3 et contrôle le volume ainsi que le mode d'exécution.
METRICS_DIR = LOCAL / 'metrics'
subprocess.run(['aws', 's3', 'cp', OUTPUT_URI + '/metrics/', str(METRICS_DIR), '--recursive'], check=True)
assert (METRICS_DIR / '_SUCCESS').exists()
metrics = json.loads(next(METRICS_DIR.glob('part-*')).read_text())
assert metrics['status'] == 'validated' and metrics['spark_master'] == 'yarn'
assert metrics['images'] == 100 and metrics['components'] == 20
assert metrics['active_partitions'] >= 2
print(json.dumps(metrics, indent=2))
if len(metrics['worker_hosts']) < 2:
    print('Attention : une seule machine observée. Vérifier executors et placement YARN avant de conclure à du multi-machine.')


In [ ]:
# Conserve les journaux et la configuration sur S3 pour la préparation de la soutenance.
# Ces preuves restent privées dans le bucket du projet ; vérifier les accès IAM configurés.
subprocess.run(['aws', 's3', 'cp', str(LOCAL), f's3://{P9_BUCKET}/evidence/{RUN_ID}/', '--recursive'], check=True)
print('Preuves :', f's3://{P9_BUCKET}/evidence/{RUN_ID}/')
print('Résultats :', OUTPUT_URI)


## Fin de session

Enregistrer ce notebook avec ses sorties et le déposer dans le préfixe de preuves. Conserver les captures de Spark UI et vérifier l’arrêt effectif du cluster suivant le plan AWS. Ce notebook n’arrête pas automatiquement un cluster interactif : ne pas fermer simplement l’onglet en supposant que la facturation s’arrête.

Les métriques ci-dessus portent sur 100 images. Augmenter progressivement le volume après ce test, et analyser les différences de temps et de ressources avant de conclure au passage à l’échelle.